# 07 — Leak Analysis & Clean Evaluation

The Kaggle dataset `vipoooool/new-plant-diseases-dataset` is offline-augmented:
each source leaf is duplicated multiple times under rotations / flips / color
perturbations, then split into `train/` and `valid/`. This means augmented
copies of the **same physical leaf** can end up on both sides of the train/test
boundary — a form of data leakage.

This notebook does two things:

1. **Step B — Leak quantification.** Strips augmentation suffixes from every
   filename (`_90deg`, `_flipLR`, `_new30degFlipLR`, ...) to recover the
   *source identity*, then measures how many test source IDs also appear in
   train.
2. **Step C — Clean evaluation.** Re-evaluates V2 / V3 / V4 (without
   re-training) on three test subsets:

   - `FULL`   – the published 8,795-image test set.
   - `LEAKED` – test images whose source IS also in train (5,732 imgs).
   - `CLEAN`  – test images whose source is NOT in train (3,063 imgs).
   - `ORIG`   – test images without any augmentation suffix (4,598 imgs).

Outputs:
- `results/metrics/leak_quantification.json` + `leak_per_class.csv`
- `results/metrics/clean_test_metrics.json`
- `results/plots/leak_per_class.png`
- `results/plots/clean_vs_full_test.png`


In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────
import json, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

PROJECT_ROOT = Path('..').resolve()
DATA_DIR    = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR  = PROJECT_ROOT / 'results' / 'models'
METRICS_DIR = PROJECT_ROOT / 'results' / 'metrics'
PLOTS_DIR   = PROJECT_ROOT / 'results' / 'plots'

if torch.backends.mps.is_available():     DEVICE = torch.device('mps')
elif torch.cuda.is_available():           DEVICE = torch.device('cuda')
else:                                     DEVICE = torch.device('cpu')

print(f'Device: {DEVICE}')


## Step B — Leak Quantification

### 2.1 Source-ID extraction

The 15 augmentation suffixes observed in the dataset are:

```
_90deg, _180deg, _270deg, _flipLR, _flipTB,
_new30degFlipLR, _new30degFlipTB,
_new90degFlipLR, _new90degFlipTB,
_new200degFlipLR, _new200degFlipTB,
_newPixel25, _newGRR, _newGGR, _CG1
```

Stripping any such suffix from a filename yields the *source identifier* —
the identity of the underlying physical leaf, independent of augmentation.

In [ ]:
# ── Cell 2: Source-ID regex ────────────────────────────────────────
AUG_SUFFIX_RE = re.compile(
    r'_(?:\d+deg(?:Flip(?:LR|TB))?'
    r'|new\d+deg(?:Flip(?:LR|TB))?'
    r'|flipLR|flipTB'
    r'|newPixel\d+'
    r'|new[GR]+R'
    r'|CG\d+)$',
    re.IGNORECASE
)

def source_id(filename: str) -> str:
    return AUG_SUFFIX_RE.sub('', Path(filename).stem)

# Smoke-test
samples = [
    '01a66316...___FREC_Scab 3003_270deg.JPG',
    '01a66316...___FREC_Scab 3003.JPG',
    'RS_Rust 1563_flipLR.JPG',
    'RS_Rust 1563.JPG',
]
for s in samples:
    print(f'  {s:<50} -> {source_id(s)}')


In [ ]:
# ── Cell 3: Scan splits ────────────────────────────────────────────
def scan(split: str) -> pd.DataFrame:
    rows = []
    for cls_dir in sorted((DATA_DIR / split).iterdir()):
        if not cls_dir.is_dir(): continue
        for img in cls_dir.iterdir():
            if img.suffix.lower() not in {'.jpg', '.jpeg', '.png'}: continue
            sid = source_id(img.name)
            rows.append({
                'split': split, 'class': cls_dir.name,
                'filename': img.name, 'source_id': sid,
                'is_augmented': Path(img.name).stem != sid,
            })
    return pd.DataFrame(rows)

df_train = scan('train')
df_val   = scan('val')
df_test  = scan('test')

for name, df in [('train', df_train), ('val', df_val), ('test', df_test)]:
    n_aug = int(df['is_augmented'].sum()); n_orig = len(df) - n_aug
    print(f'  {name:<5}: {len(df):>6,} imgs | '
          f'{df["source_id"].nunique():>5,} sources | '
          f'orig={n_orig:>5,} aug={n_aug:>5,}')


In [ ]:
# ── Cell 4: Global leak metrics ────────────────────────────────────
def overlap(a, b):
    sa = set(a['source_id']); sb = set(b['source_id']); sh = sa & sb
    return {
        'sources_a': len(sa), 'sources_b': len(sb),
        'shared_sources': len(sh),
        'pct_b_sources_in_a': round(100 * len(sh) / max(1, len(sb)), 2),
    }

global_leak = {
    'train_vs_val':  overlap(df_train, df_val),
    'train_vs_test': overlap(df_train, df_test),
    'val_vs_test':   overlap(df_val,   df_test),
}
print(json.dumps(global_leak, indent=2))


In [ ]:
# ── Cell 5: Per-class leak ─────────────────────────────────────────
train_sources_by_class = {c: set(df_train.loc[df_train['class']==c, 'source_id'])
                          for c in df_train['class'].unique()}

per_class = []
for cls in sorted(df_test['class'].unique()):
    tr = train_sources_by_class.get(cls, set())
    te_rows = df_test[df_test['class'] == cls]
    te_src  = set(te_rows['source_id'])
    sh      = te_src & tr
    leak_imgs = te_rows[te_rows['source_id'].isin(sh)]
    per_class.append({
        'class': cls,
        'test_imgs': len(te_rows),
        'test_sources': len(te_src),
        'shared_sources': len(sh),
        'pct_test_sources_leaked': round(100*len(sh)/max(1,len(te_src)), 2),
        'pct_test_images_leaked':  round(100*len(leak_imgs)/max(1,len(te_rows)), 2),
    })
df_leak = pd.DataFrame(per_class)

METRICS_DIR.mkdir(parents=True, exist_ok=True)
df_leak.to_csv(METRICS_DIR / 'leak_per_class.csv', index=False)
print(df_leak.to_string(index=False))


In [ ]:
# ── Cell 6: Identify clean / non-augmented test subsets ────────────
train_sources_all = set(df_train['source_id'])
df_test_clean = df_test[~df_test['source_id'].isin(train_sources_all)]
df_test_orig  = df_test[~df_test['is_augmented']]

print(f'CLEAN  test (source NOT in train): {len(df_test_clean):>5,} '
      f'({100*len(df_test_clean)/len(df_test):.1f}%)')
print(f'ORIG   test (no aug suffix):       {len(df_test_orig):>5,} '
      f'({100*len(df_test_orig)/len(df_test):.1f}%)')

leak_summary = {
    'evidence': 'vipoooool/new-plant-diseases-dataset is offline-augmented (15 distinct suffixes).',
    'global': global_leak,
    'totals': {
        'train_images': len(df_train), 'val_images': len(df_val), 'test_images': len(df_test),
        'train_sources': df_train['source_id'].nunique(),
        'val_sources':   df_val['source_id'].nunique(),
        'test_sources':  df_test['source_id'].nunique(),
    },
    'clean_test_subset': {'size': len(df_test_clean),
                          'pct_of_test': round(100*len(df_test_clean)/len(df_test), 2)},
    'non_augmented_test_subset': {'size': len(df_test_orig),
                                  'pct_of_test': round(100*len(df_test_orig)/len(df_test), 2)},
}
with open(METRICS_DIR / 'leak_quantification.json', 'w') as f:
    json.dump(leak_summary, f, indent=2)
print(f"\nSaved: {METRICS_DIR / 'leak_quantification.json'}")


In [ ]:
# ── Cell 7: Leak-per-class plot ────────────────────────────────────
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(10, 11))
ordered = df_leak.sort_values('pct_test_sources_leaked')
ax.barh(ordered['class'], ordered['pct_test_sources_leaked'],
        color=['#d62728' if v>50 else '#2ca02c' for v in ordered['pct_test_sources_leaked']],
        edgecolor='black', linewidth=0.5)
ax.set_xlim(0, 105)
ax.set_xlabel('% of test source IDs already present in train')
ax.set_title('Data-leak quantification per class\n'
             'vipoooool/new-plant-diseases-dataset (offline-augmented)',
             fontweight='bold')
for i, v in enumerate(ordered['pct_test_sources_leaked']):
    ax.text(v+1, i, f'{v:.0f}%', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'leak_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {PLOTS_DIR / 'leak_per_class.png'}")


## Step C — Clean Evaluation

Re-evaluate V2 / V3 / V4 (no re-training) on FULL / LEAKED / CLEAN / ORIG subsets
of the test set. V4 uses the cached DINOv3 embeddings, so the linear probe is
re-fit in a few seconds. The masks are built against the deterministic
`ImageFolder.samples` order to guarantee alignment with the published metrics.


In [ ]:
# ── Cell 8: Test loader + alignment masks ──────────────────────────
IMG_SIZE = 224; BATCH = 64
IMAGENET_MEAN = [0.485, 0.456, 0.406]; IMAGENET_STD = [0.229, 0.224, 0.225]

tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
test_ds = datasets.ImageFolder(DATA_DIR / 'test', transform=tf)
num_classes = len(test_ds.classes)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False)
print(f'test: {len(test_ds):,} imgs | classes: {num_classes}')

n = len(test_ds.samples)
clean_mask  = np.zeros(n, dtype=bool)
orig_mask   = np.zeros(n, dtype=bool)
for i, (path, _) in enumerate(test_ds.samples):
    name = Path(path).name; sid = source_id(name)
    clean_mask[i] = sid not in train_sources_all
    orig_mask[i]  = Path(name).stem == sid
leaked_mask = ~clean_mask

print(f'CLEAN : {clean_mask.sum():>5,} | LEAKED: {leaked_mask.sum():>5,} | ORIG: {orig_mask.sum():>5,}')


In [ ]:
# ── Cell 9: Helpers ────────────────────────────────────────────────
@torch.no_grad()
def predict_torch(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        preds.append(model(x).argmax(1).cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(preds), np.concatenate(labels)

def metrics_on(mask, preds, labels):
    if mask.sum() == 0: return {'n': 0}
    y_true, y_pred = labels[mask], preds[mask]
    return {
        'n': int(mask.sum()),
        'accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 4),
        'recall':    round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 4),
        'f1':        round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 4),
    }


In [ ]:
# ── Cell 10: V2 — Custom CNN ───────────────────────────────────────
class _ConvBlock(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ic, oc, 3, padding=1, bias=False),
            nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class PlantDiseaseNet(nn.Module):
    def __init__(self, nc):
        super().__init__()
        self.features = nn.Sequential(
            _ConvBlock(3,32),   nn.MaxPool2d(2),
            _ConvBlock(32,64),  nn.MaxPool2d(2),
            _ConvBlock(64,128), nn.MaxPool2d(2),
            _ConvBlock(128,256),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(256,512), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512,256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256,nc),
        )
    def forward(self, x):
        x = self.features(x); x = self.pool(x).flatten(1)
        return self.classifier(x)

m2 = PlantDiseaseNet(num_classes).to(DEVICE)
ck = torch.load(MODELS_DIR / 'v2_custom_cnn' / 'checkpoint.pth', map_location=DEVICE, weights_only=False)
m2.load_state_dict(ck['model_state_dict'])
t0 = time.time(); preds2, labels2 = predict_torch(m2, test_loader)
print(f'V2 inference: {time.time()-t0:.1f}s')
del m2

v2 = {
    'full':          metrics_on(np.ones_like(clean_mask), preds2, labels2),
    'clean':         metrics_on(clean_mask, preds2, labels2),
    'leaked':        metrics_on(leaked_mask, preds2, labels2),
    'non_augmented': metrics_on(orig_mask, preds2, labels2),
}
print(json.dumps(v2, indent=2))


In [ ]:
# ── Cell 11: V3 — ResNet50 Transfer Learning ───────────────────────
def build_v3(nc):
    net = models.resnet50(weights=None)
    net.fc = nn.Sequential(
        nn.Linear(net.fc.in_features, 512), nn.ReLU(inplace=True), nn.Dropout(0.5),
        nn.Linear(512, nc),
    )
    return net

m3 = build_v3(num_classes).to(DEVICE)
ck = torch.load(MODELS_DIR / 'v3_transfer_learning' / 'checkpoint.pth', map_location=DEVICE, weights_only=False)
m3.load_state_dict(ck['model_state_dict'])
t0 = time.time(); preds3, labels3 = predict_torch(m3, test_loader)
print(f'V3 inference: {time.time()-t0:.1f}s')
del m3

v3 = {
    'full':          metrics_on(np.ones_like(clean_mask), preds3, labels3),
    'clean':         metrics_on(clean_mask, preds3, labels3),
    'leaked':        metrics_on(leaked_mask, preds3, labels3),
    'non_augmented': metrics_on(orig_mask, preds3, labels3),
}
print(json.dumps(v3, indent=2))


In [ ]:
# ── Cell 12: V4 — DINOv3 + Linear Probe (cached embeddings) ────────
slug = 'dinov3-vitb16-pretrain-lvd1689m'
EMB = MODELS_DIR / 'v4_dinov3_probe' / 'embeddings'
tr_npz = np.load(EMB / f'train__{slug}.npz')
te_npz = np.load(EMB / f'test__{slug}.npz')
Xtr = normalize(tr_npz['X'], norm='l2'); ytr = tr_npz['y']
Xte = normalize(te_npz['X'], norm='l2'); yte = te_npz['y']
print(f'embeddings train: {Xtr.shape} | test: {Xte.shape} (L2-normalized)')

t0 = time.time()
clf = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
clf.fit(Xtr, ytr)
print(f'V4 linear-probe fit: {time.time()-t0:.1f}s')
preds4 = clf.predict(Xte); labels4 = yte

if not np.array_equal(labels4, labels2):
    raise RuntimeError('V4 cached label order differs from torch DataLoader order — '
                       'regenerate the embedding cache from notebook 04.')

v4 = {
    'full':          metrics_on(np.ones_like(clean_mask), preds4, labels4),
    'clean':         metrics_on(clean_mask, preds4, labels4),
    'leaked':        metrics_on(leaked_mask, preds4, labels4),
    'non_augmented': metrics_on(orig_mask, preds4, labels4),
}
print(json.dumps(v4, indent=2))


In [ ]:
# ── Cell 13: Save unified clean_test_metrics.json + plot ───────────
results = {
    'v2': v2, 'v3': v3, 'v4': v4,
    'meta': {
        'device': str(DEVICE),
        'test_size':          int(len(test_ds)),
        'clean_size':         int(clean_mask.sum()),
        'leaked_size':        int(leaked_mask.sum()),
        'non_augmented_size': int(orig_mask.sum()),
        'definitions': {
            'full':          'Entire test set (current published metric).',
            'clean':         'Test images whose source_id is NOT in train. Most rigorous.',
            'leaked':        'Test images whose source_id IS in train. Reflects the leak.',
            'non_augmented': 'Test images without augmentation suffix in filename.',
        },
    },
}
with open(METRICS_DIR / 'clean_test_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"Saved: {METRICS_DIR / 'clean_test_metrics.json'}")

versions = ['v2', 'v3', 'v4']
subsets  = ['full', 'leaked', 'non_augmented', 'clean']
colors   = {'full': '#888', 'leaked': '#d62728',
            'non_augmented': '#ff7f0e', 'clean': '#2ca02c'}

fig, ax = plt.subplots(figsize=(11, 6))
width = 0.2; x = np.arange(len(versions))
for i, s in enumerate(subsets):
    vals = [results[v][s].get('accuracy', 0) for v in versions]
    ax.bar(x + (i-1.5)*width, vals, width, label=s.upper().replace('_','-'),
           color=colors[s], edgecolor='black', linewidth=0.5)
    for j, v in enumerate(vals):
        if v:
            ax.text(x[j] + (i-1.5)*width, v + 0.003, f'{v:.3f}',
                    ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(['V2 CustomCNN', 'V3 ResNet50', 'V4 DINOv3'])
ax.set_ylabel('Accuracy'); ax.set_ylim(0.94, 1.005)
ax.set_title('Test accuracy by subset — leak isolation\n'
             f'CLEAN = source NOT in train ({int(clean_mask.sum()):,} imgs) | '
             f'LEAKED = source IN train ({int(leaked_mask.sum()):,})',
             fontweight='bold')
ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'clean_vs_full_test.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {PLOTS_DIR / 'clean_vs_full_test.png'}")


## Conclusion

For all three deep models the **clean** accuracy is within ±0.1 points of the
published full-test accuracy. For V3 and V4 the clean number is even marginally
*higher* than full. Interpretation:

- The offline augmentations (rotations, flips, color shifts) are precisely the
  invariances that the deep models are designed to learn. Seeing a 90°-rotated
  copy of a leaf in train does not give the model an "answer key" for the test
  set — the model has to recognize the underlying leaf either way.
- The 63% source leak is therefore **statistically irrelevant** for the published
  accuracies on this task.
- The methodological caveat in the report is now *quantified*: the absolute
  performance numbers are robust to the leak, not inflated by it.